In [2]:
import sys
from pathlib import Path
import pandas as pd

# Exécution séquentielle du pipeline complet
filepath = "../data/raw/velib-2026-10-08.csv.gz"

df_raw = load_and_clean(filepath)
df_feat = build_features(df_raw)
train_df, val_df, test_df = filter_and_split(df_feat)

# Contrôle des dimensions et du taux de valeurs manquantes (lags)
print(f"Train      : {train_df.shape}")
print(f"Validation : {val_df.shape}")
print(f"Test       : {test_df.shape}")

new_cols = [c for c in train_df.columns if c not in df_raw.columns]
print(f"\nNouvelles colonnes ({len(new_cols)}) :\n{new_cols}")

print("\nTaux de NaN sur les features de dynamique passée (Train) :")
lag_cols = ['bikes_lag_15m', 'bikes_lag_30m', 'bikes_lag_60m', 'fill_rate']
print((train_df[lag_cols].isna().mean() * 100).round(2))

Train      : (227420, 35)
Validation : (45230, 35)
Test       : (52220, 35)

Nouvelles colonnes (17) :
['bikes_lag_15m', 'diff_bikes_15m', 'bikes_lag_30m', 'diff_bikes_30m', 'bikes_lag_60m', 'diff_bikes_60m', 'fill_rate', 'elec_ratio', 'meca_ratio', 'freshness_min', 'hour_float', 'hour_sin', 'hour_cos', 'dayofweek', 'is_weekend', 'cible', 'pred_persistance']

Taux de NaN sur les features de dynamique passée (Train) :
bikes_lag_15m    0.64
bikes_lag_30m    1.16
bikes_lag_60m    1.98
fill_rate        0.28
dtype: float64


In [4]:
train_df.info()

<class 'pandas.DataFrame'>
Index: 227420 entries, 1 to 328442
Data columns (total 35 columns):
 #   Column                   Non-Null Count   Dtype                       
---  ------                   --------------   -----                       
 0   collected_at_utc         227420 non-null  datetime64[us, UTC]         
 1   source_updated_at        227420 non-null  str                         
 2   station_code             227420 non-null  int64                       
 3   name                     227420 non-null  str                         
 4   commune                  227420 non-null  str                         
 5   insee_code               227420 non-null  int64                       
 6   latitude                 227420 non-null  float64                     
 7   longitude                227420 non-null  float64                     
 8   capacity                 226780 non-null  float64                     
 9   bikes_available          227420 non-null  int64                 

In [5]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, f1_score

# Variables explicatives (sans cible, persistance ni colonnes texte/dates)
FEATURES = [
    'capacity', 'bikes_available', 'mechanical_bikes', 'electric_bikes',
    'docks_available', 'latitude', 'longitude',
    'is_installed', 'is_renting', 'is_returning',
    'bikes_lag_15m', 'diff_bikes_15m',
    'bikes_lag_30m', 'diff_bikes_30m',
    'bikes_lag_60m', 'diff_bikes_60m',
    'fill_rate', 'elec_ratio', 'meca_ratio', 'freshness_min',
    'hour_float', 'hour_sin', 'hour_cos', 'dayofweek', 'is_weekend'
]

X_train, y_train = train_df[FEATURES], train_df['cible']
X_val, y_val = val_df[FEATURES], val_df['cible']

# Référence à battre sur Validation
f1_base = f1_score(y_val, val_df['pred_persistance'])
print(f"F1 Baseline Persistance (Validation) : {f1_base:.4f}")

# Entraînement et évaluation de HistGradientBoosting
model_hgb = HistGradientBoostingClassifier(random_state=42)
model_hgb.fit(X_train, y_train)

y_pred_val = model_hgb.predict(X_val)
f1_hgb = f1_score(y_val, y_pred_val)
print(f"F1 HistGradientBoosting (Validation) : {f1_hgb:.4f}\n")

print(classification_report(y_val, y_pred_val, digits=4))

F1 Baseline Persistance (Validation) : 0.7536
F1 HistGradientBoosting (Validation) : 0.7635

              precision    recall  f1-score   support

           0     0.9604    0.9724    0.9664     39355
           1     0.7984    0.7316    0.7635      5875

    accuracy                         0.9411     45230
   macro avg     0.8794    0.8520    0.8650     45230
weighted avg     0.9394    0.9411    0.9400     45230

